<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/MiniProjet_W9_D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Partie 1 : Installation des dépendances
Cette cellule installe les bibliothèques nécessaires pour LangChain, LangGraph, l'intégration Gemini et le protocole MCP.

In [1]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "fastmcp>=2.0.0" \
  "nest_asyncio" \
  "mcp-server-git"

# Restart kernel to ensure imports are available
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.2/765.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.1/230.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.0/273.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

{'status': 'ok', 'restart': True}

### Partie 2 : Configuration de la GOOGLE_API_KEY
Vous pouvez configurer votre clé de deux manières :
1. **Secrets Colab (Recommandé)** : Cliquez sur l'icône de clé (🔑) à gauche, ajoutez `GOOGLE_API_KEY`.
2. **Variable d'environnement** : Définie directement dans le code ci-dessous.

In [ ]:
import os
from google.colab import userdata
import nest_asyncio

nest_asyncio.apply()

try:
    key = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = key
    print("✅ Clé API configurée via les secrets.")
except Exception:
    print("⚠️ Secret 'GOOGLE_API_KEY' non trouvé.")
    # Fallback pour saisie manuelle si le secret n'existe pas
    if not os.environ.get("GOOGLE_API_KEY"):
        from getpass import getpass
        os.environ["GOOGLE_API_KEY"] = getpass("Veuillez saisir votre GOOGLE_API_KEY : ")

⚠️ Secret 'GOOGLE_API_KEY' non trouvé.


### Partie 3 : Vérification de Node.js et NPM
Les serveurs MCP tiers (comme filesystem) utilisent souvent `npx`. Nous vérifions et installons Node.js si absent.

In [2]:
import subprocess

def check_node():
    try:
        node_version = subprocess.check_output(["node", "--version"]).decode().strip()
        npx_version = subprocess.check_output(["npx", "--version"]).decode().strip()
        print(f"✅ Node.js {node_version} et NPM/NPX {npx_version} sont disponibles.")
    except FileNotFoundError:
        print("❌ Node.js non trouvé. Installation en cours...")
        !apt-get -qq update
        !apt-get -qq install -y nodejs npm
        print("✅ Installation terminée.")

check_node()

✅ Node.js v20.19.0 et NPM/NPX 10.8.2 sont disponibles.


### Partie 4 : Initialisation du Workspace et Git
Nous créons un répertoire de travail et initialisons un dépôt Git avec des fichiers de test.

In [3]:
import os
WORKDIR = "/content/workspace"
os.makedirs(WORKDIR, exist_ok=True)
%cd {WORKDIR}

# Re-initialisation du repo
!git init
!git config --global user.email "assistant@example.com"
!git config --global user.name "AI Assistant"

with open("README.md", "w") as f:
    f.write("# AI Workspace Project\nCeci est un dépôt de test pour l'assistant MCP.")

with open("main.py", "w") as f:
    f.write("def hello():\n    print('Hello World')\n\nif __name__ == '__main__':\n    hello()")

!git add .
!git commit -m "Initial commit"
print(f"✅ Workspace restauré dans {WORKDIR}")

/content/workspace
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/workspace/.git/
[master (root-commit) 22c0ea4] Initial commit
 2 files changed, 7 insertions(+)
 create mode 100644 README.md
 create mode 100644 main.py
✅ Workspace restauré dans /content/workspace


### Partie 5 : Création du serveur MCP personnalisé (custom_ops)
Ce serveur utilise `FastMCP` pour exposer des outils spécifiques de traitement de texte et de statistiques.

In [4]:
from pathlib import Path
import textwrap

server_script_path = Path("/content/custom_mcp_server.py")
server_script_path.write_text(textwrap.dedent("""
from fastmcp import FastMCP
from typing import List, Dict
import os

mcp = FastMCP(name="custom_ops")

@mcp.tool
def ping() -> str:
    \"\"\"Outil de vérification de santé.\"\"\"
    return "pong"

@mcp.tool
def summarize_lines(lines: List[str]) -> Dict[str, int]:
    \"\"\"Compte les lignes totales et non vides.\"\"\"
    return {"total": len(lines), "non_empty": sum(1 for l in lines if l.strip())}

@mcp.tool
def count_words(text: str) -> int:
    \"\"\"Compte le nombre de mots dans un texte.\"\"\"
    return len(text.split())

@mcp.tool
def extract_headings(markdown: str) -> List[str]:
    \"\"\"Extrait les titres (H1, H2) d'un texte Markdown.\"\"\"
    return [l for l in markdown.splitlines() if l.startswith('#')]

@mcp.tool
def project_statistics(folder: str) -> Dict[str, int]:
    \"\"\"Calcule des statistiques simples sur les fichiers du dossier.\"\"\"
    files = os.listdir(folder)
    return {"file_count": len(files), "python_files": len([f for f in files if f.endswith('.py')])}

if __name__ == '__main__':
    mcp.run(transport='stdio')
"""), encoding="utf-8")

print(f"✅ Serveur personnalisé écrit dans {server_script_path}")

✅ Serveur personnalisé écrit dans /content/custom_mcp_server.py


### Partie 12 : Génération automatique du README.md
Cette cellule génère le fichier de documentation du projet directement dans le workspace.

In [2]:
import os
WORKDIR = "/content/workspace"
os.makedirs(WORKDIR, exist_ok=True)

readme_content = """# AI Workspace Assistant

## Présentation
Système multi-agents orchestré par Gemini utilisant le Model Context Protocol (MCP).

## Architecture
- **LLM**: Gemini 1.5 Flash
- **Orchestration**: LangGraph (Agentic Workflow)
- **Outils**: MultiServerMCPClient
- **Serveurs MCP**:
  1. Filesystem (npx)
  2. Git (python)
  3. Custom Ops (FastMCP)
"""

# Ecriture du fichier README
with open(os.path.join(WORKDIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme_content)

print(f"✅ README.md créé avec succès dans {WORKDIR}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/workspace/README.md'

### Partie 13 & 14 : Démonstrations finales et Résumé
Exécution des cas d'usage Git et Markdown pour prouver l'intégration multi-serveurs.

In [ ]:
# Exemple 3 : Historique Git
await run_demo("Generate a summary of the latest Git commits and suggest a changelog.")

# Exemple 4 : Analyse Markdown
await run_demo("Extract headings from README.md and count the words in that file.")

### Résumé de l'exécution

**Configuration des serveurs** :
- **filesystem** : Accès direct aux fichiers de `/content/workspace`
- **git** : Gestion du dépôt local
- **custom_ops** : Fonctions métier spécifiques (FastMCP)

**Flux de travail (Workflow)** :
1. **Input** : Requête utilisateur.
2. **Raisonnement** : Gemini décide quel serveur MCP interroger.
3. **Action** : Appel via `MultiServerMCPClient`.
4. **Synthèse** : Gemini agrège les résultats pour la réponse finale.

### Partie 6 : Connexion aux serveurs MCP
Nous utilisons `MultiServerMCPClient` pour regrouper les outils des serveurs Filesystem (npx), Git (Python) et Custom (Python).

In [ ]:
import asyncio
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

# Redéfinition nécessaire après redémarrage
WORKDIR = "/content/workspace"
server_script_path = "/content/custom_mcp_server.py"

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
    "custom_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_script_path)],
    }
}

async def initialize_tools():
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
    tools = await client.get_tools()
    return client, tools

try:
    client, all_tools = asyncio.run(initialize_tools())
    print(f"✅ Nombre total d'outils chargés : {len(all_tools)}")
except Exception as e:
    print(f"❌ Erreur lors du chargement : {e}")

### Partie 7 & 8 : Création de l'Agent Gemini avec LangGraph
L'agent utilise `ChatGoogleGenerativeAI`. Le graphe permet une boucle de rétroaction : Gemini analyse la question, appelle les outils si nécessaire, traite les résultats et répond.

In [ ]:
import os
from typing import Annotated, Sequence, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages

# Ensure the key is used directly if environment sync fails
api_key = os.environ.get("GOOGLE_API_KEY")

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Initialize model with explicit key check
if not api_key:
    raise ValueError("GOOGLE_API_KEY manquante. Veuillez la configurer dans les secrets.")

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", google_api_key=api_key, temperature=0)
bound_llm = llm.bind_tools(all_tools)

def call_model(state: AgentState):
    response = bound_llm.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(all_tools))
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, ["tools", END])
workflow.add_edge("tools", "agent")

app = workflow.compile()
print("✅ Graphe LangGraph prêt avec authentification Gemini.")

### Partie 9 & 10 : Démonstrations et Affichage des Tool Calls
Voici une fonction utilitaire pour exécuter les requêtes et afficher le raisonnement de l'agent.

In [ ]:
from langchain_core.messages import HumanMessage
import asyncio

async def run_demo(query: str):
    if 'app' not in globals():
        print("❌ L'agent (app) n'est pas initialisé. Vérifiez la cellule précédente.")
        return

    print(f"\n🚀 Question : {query}")
    print("-" * 50)

    inputs = {"messages": [HumanMessage(content=query)]}

    try:
        async for event in app.astream(inputs, stream_mode="values"):
            message = event["messages"][-1]

            if hasattr(message, "tool_calls") and message.tool_calls:
                for tc in message.tool_calls:
                    print(f"🧠 Pensée : Utilisation de {tc['name']}")

            elif message.type == "ai" and not message.tool_calls:
                print(f"\n🏁 Réponse finale :\n{message.content}")
    except Exception as e:
        print(f"❌ Erreur lors de l'exécution : {e}")

# Exemple 1
if 'app' in globals():
    await run_demo("Summarize this project based on the files present in the workspace.")

### Partie 11 : Gestion des erreurs
L'agent est capable de gérer les erreurs grâce à la boucle de LangGraph. Si un outil échoue, le message d'erreur est renvoyé au LLM qui peut tenter une autre approche.

In [ ]:
# Exemple 2 : Utilisation d'outils combinés
await run_demo("Find every Python file then count the words in each and give me the total statistics.")